In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_DimTime")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
.getOrCreate())

In [ ]:
gold_dim_time_path = "../../data_lake/gold/dim_time"

In [ ]:
seconds_in_day = 24 * 60 * 60

df_dim_time = (spark.range(0, seconds_in_day)
    .withColumnRenamed("id", "second_of_day")
    .withColumn("hour", F.floor(F.col("second_of_day") / 3600).cast("int"))
    .withColumn("minute", F.floor((F.col("second_of_day") % 3600) / 60).cast("int"))
    .withColumn("second", (F.col("second_of_day") % 60).cast("int"))
    .withColumn("time_key", F.col("second_of_day").cast("int"))
    .withColumn("hour_12", F.expr("CASE WHEN hour % 12 = 0 THEN 12 ELSE hour % 12 END"))
    .withColumn("am_pm", F.when(F.col("hour") < 12, F.lit("AM")).otherwise(F.lit("PM")))
    .withColumn("time_24", F.format_string("%02d:%02d:%02d", F.col("hour"), F.col("minute"), F.col("second")))
    .withColumn("time_12", F.format_string("%02d:%02d:%02d %s", F.col("hour_12"), F.col("minute"), F.col("second"), F.col("am_pm")))
    .withColumn("hour_bucket", F.format_string("%02d:00-%02d:59", F.col("hour"), F.col("hour")))
    .withColumn("part_of_day",
        F.when((F.col("hour") >= 5) & (F.col("hour") < 12), F.lit("Morning"))
         .when((F.col("hour") >= 12) & (F.col("hour") < 17), F.lit("Afternoon"))
         .when((F.col("hour") >= 17) & (F.col("hour") < 21), F.lit("Evening"))
         .otherwise(F.lit("Night"))
    )
    .withColumn("gold_timestamp", F.current_timestamp())
    .select(
        "time_key",
        "hour",
        "minute",
        "second",
        "second_of_day",
        "hour_12",
        "am_pm",
        "time_24",
        "time_12",
        "hour_bucket",
        "part_of_day",
        "gold_timestamp"
    )
)

In [ ]:
df_dim_time.orderBy("time_key").show(10, truncate=False)
df_dim_time.count()

In [ ]:
df_dim_time.write.mode("overwrite").format("parquet").save(gold_dim_time_path)

In [ ]:
spark.stop()